## Minimal notebook: NN vs PSMC (humanlike)

This notebook reproduces the **same core logic** as `psmc_checker/plot_runs_with_nn.py` on one existing synthetic run:

- input: `psmc_checker/runs/hl/exp1_hl.psmcfa` and `exp1_hl.psmc`
- NN: GRU checkpoint `test_all_v1/model/gru_set3_ce.ckpt`

Key point: **the NN curve is produced with no extra manipulations**:

1) run model → `logits`
2) convert to probabilities: `softmax(logits)`
3) aggregate over all positions: `sum over (batch, position)`
4) normalize to a distribution (`priors`)
5) convert `priors → Ne` with the same formula as `mark_all.py` (as implemented in `plot_runs_with_nn.py`)

Finally we plot **PSMC vs TRUE vs NN** and compute relative-error metrics.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

from psmc_checker import plot_results
from psmc_checker.plot_runs_with_nn import (
    _build_gru_model,
    _load_checkpoint_state_dict,
    _load_windows_from_psmcfa,
    _relative_metrics,
    _true_ne_at_times,
)


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for p in [start, *start.parents]:
        if (p / "deepgene").exists() and (p / "psmc_checker").exists():
            return p
    raise FileNotFoundError("Cannot locate repo root (expected folders: deepgene/ and psmc_checker/)")


ROOT = find_repo_root()

# Inputs for this minimal demo
psmc_path = ROOT / "psmc_checker" / "runs" / "hl" / "exp1_hl.psmc"
psmcfa_path = ROOT / "psmc_checker" / "runs" / "hl" / "exp1_hl.psmcfa"
ckpt_path = ROOT / "test_all_v1" / "model" / "gru_set3_ce.ckpt"

assert psmc_path.exists(), psmc_path
assert psmcfa_path.exists(), psmcfa_path
assert ckpt_path.exists(), ckpt_path

# PSMC scaling parameters.
# NOTE: the run metadata CSV (psmc_checker/runs/hl/exp1_hl.csv) used mu=1.25e-07.
bin_size = 100
mu = 1.25e-7
generation_time = 25.0
psmc_time_mult = 2.0
psmc_size_mult = 1.0

# NN inference controls
device = torch.device("cpu")
batch_size = 2

print("ROOT:", ROOT)
print("PSMC:", psmc_path)
print("PSMCFA:", psmcfa_path)
print("CKPT:", ckpt_path)
print("device:", device, "batch_size:", batch_size)

In [ ]:
# 1) Load NN inputs (binary windows) from .psmcfa
#    This matches the helper logic used by plot_runs_with_nn.py

windows = _load_windows_from_psmcfa(psmcfa_path)  # shape: (n_records, L)
print("windows shape:", windows.shape, windows.dtype)
print("fraction of 1s (heterozygous bins):", float(windows.mean()))

In [ ]:
# 2) Run GRU and aggregate priors
#    IMPORTANT: this is the entire "NN output" logic.
#    There is NO smoothing / calibration here.
#
#    priors[c] = sum_{all positions} softmax(logits)[c]
#    then normalized to sum(priors)=1

state_dict = _load_checkpoint_state_dict(ckpt_path)
model = _build_gru_model(state_dict)
model.load_state_dict(state_dict, strict=True)
model.to(device)
model.eval()

expects_long = isinstance(getattr(model, "embedding", None), torch.nn.Embedding)

# infer number of classes from checkpoint
n_class = int(state_dict["predictor.dense3.weight"].shape[0])
priors_sum = torch.zeros((n_class,), dtype=torch.float64)

total_positions = 0
with torch.no_grad():
    for i in range(0, int(windows.shape[0]), int(batch_size)):
        batch = windows[i : i + int(batch_size)]
        xb = torch.from_numpy(batch)
        xb = xb.long() if expects_long else xb.float()
        xb = xb.to(device)

        logits = model(xb)  # (B, L, C)
        probs = torch.softmax(logits, dim=-1)  # (B, L, C)

        # sum over batch and positions -> (C,)
        priors_sum += probs.sum(dim=(0, 1)).detach().cpu().double()
        total_positions += int(probs.shape[0] * probs.shape[1])

priors = (priors_sum / priors_sum.sum()).numpy().astype(np.float64)

print("n_class:", n_class)
print("total_positions:", total_positions)
print("priors sum:", float(priors.sum()))
print("top-10 classes by prior:")
for idx in np.argsort(-priors)[:10]:
    print(f"  class={int(idx):2d} prior={priors[idx]:.6f}")

In [ ]:
# 3) Convert priors -> Ne
#    This matches mark_all.py behavior (as implemented in plot_runs_with_nn.py):
#    - tau = limits[i+1] - limits[i]
#    - use priors[j] for j in [0..len(tau)-1], ignore any extra tail in priors

import math

limits = list(getattr(plot_results.dgn, "limits"))


def probability_to_N_mark_all(priors: np.ndarray, limits: list[float]) -> list[float]:
    tau = np.asarray(limits[1:], dtype=float) - np.asarray(limits[:-1], dtype=float)
    pri = np.asarray(priors, dtype=float)
    if pri.shape[0] < tau.shape[0]:
        raise ValueError(f"Expected priors length >= {tau.shape[0]}, got {pri.shape[0]}")

    l = 1.0
    out: list[float] = []
    eps = 1e-12
    for j in range(int(tau.shape[0])):
        pj = float(pri[j])
        # guard against log(<=0)
        val = 1.0 - pj / l
        val = max(min(val, 1.0 - eps), eps)
        Nj = float(np.round(float(tau[j]) / (-2.0 * math.log(val))))
        out.append(Nj)
        l *= float(math.exp(-float(tau[j]) / (2.0 * Nj)))
    return out


nn_ne = probability_to_N_mark_all(priors, limits)
print("len(limits):", len(limits))
print("len(nn_ne):", len(nn_ne))
print("nn_ne[:10]:", nn_ne[:10])

In [ ]:
# 4) Build PSMC curve + TRUE curve, then compare vs NN

limits_x = [float(t) * float(generation_time) for t in limits]

# PSMC curve (scaled)
psmc_times, psmc_sizes, N0 = plot_results.psmc_parse_times_sizes(
    psmc_path,
    bin_size=bin_size,
    mutation_rate=mu,
    generation_time=generation_time,
    N0_override=None,
    time_mult=psmc_time_mult,
    size_mult=psmc_size_mult,
)

# TRUE curve (deepgene demography)
dem = plot_results.build_demography("get_humanlike", initial_size=10_000, random_seed=42)
true_ne_full = np.asarray(plot_results.demography_to_Ne(dem, limits), dtype=float).reshape(-1)
true_ne_intervals = true_ne_full[:-1]

# NOTE: deep-time tail
# `limits` has a very coarse last pre-tail interval (120k -> 10M generations).
# NN-derived Ne can become huge there (tiny prior over a massive tau), and PSMC is also unreliable at very deep times.
# For a clean "NN comparable to PSMC" demo we evaluate + plot only up to 120k generations.
x_max_years = 120_000 * generation_time  # 3,000,000 years

# ---- Metrics (restricted time range) ----

# PSMC vs TRUE on PSMC timepoints
true_at_psmc = _true_ne_at_times(limits_x=limits_x, true_ne=true_ne_full, times=psmc_times)
mask = np.asarray(psmc_times) <= float(x_max_years)
psmc_fmse, psmc_rmse, psmc_fmae, psmc_n = _relative_metrics(
    np.asarray(true_at_psmc)[mask],
    np.asarray(psmc_sizes, dtype=float)[mask],
)

# NN vs TRUE on interval grid (only intervals whose right edge <= x_max)
interval_mask = np.asarray(limits_x[1:], dtype=float) <= float(x_max_years)
nn_fmse, nn_rmse, nn_fmae, nn_n = _relative_metrics(
    np.asarray(true_ne_intervals, dtype=float)[interval_mask],
    np.asarray(nn_ne, dtype=float)[interval_mask],
)

# NN vs PSMC on PSMC timepoints (PSMC treated as reference, like 1000G mode)
nn_full = np.asarray(list(nn_ne) + [nn_ne[-1]], dtype=float)
nn_at_psmc = _true_ne_at_times(limits_x=limits_x, true_ne=nn_full, times=psmc_times)
np_fmse, np_rmse, np_fmae, np_n = _relative_metrics(
    np.asarray(psmc_sizes, dtype=float)[mask],
    np.asarray(nn_at_psmc, dtype=float)[mask],
)

print(f"Time range used: [0, {x_max_years:,.0f}] years")
print("PSMC vs TRUE (restricted):")
print("  points:", psmc_n, "FMSE:", psmc_fmse, "RMSE:", psmc_rmse, "FMAE:", psmc_fmae)
print("NN   vs TRUE (restricted):")
print("  points:", nn_n, "FMSE:", nn_fmse, "RMSE:", nn_rmse, "FMAE:", nn_fmae)
print("NN   vs PSMC (restricted):")
print("  points:", np_n, "FMSE:", np_fmse, "RMSE:", np_rmse, "FMAE:", np_fmae)

# ---- Plot (same restricted range) ----
fig, ax = plt.subplots(figsize=(8, 4))
ax.step(psmc_times, psmc_sizes, where="post", linestyle="--", color="tab:blue", label=f"PSMC (N0={N0:.2f})")
ax.step(limits_x, true_ne_full, where="post", color="tab:red", label="TRUE (get_humanlike)")
ax.step(limits_x, list(nn_ne) + [nn_ne[-1]], where="post", color="black", linestyle="-.", linewidth=1.5, label="NN")

ax.set_xlabel("Time (years)")
ax.set_ylabel("N_e")
ax.set_xlim(0, float(x_max_years))
ax.set_ylim(0, 20_000)
ax.grid(True)
ax.legend(loc="best")
plt.show()